In [1]:
pip install openai pypdf tqdm numpy


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install qdrant-client

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from openai import OpenAI
from qdrant_client import QdrantClient

# ---------- OPENAI ----------
client = OpenAI(api_key="")  # берет ключ из ENV

EMBED_MODEL = "text-embedding-3-large"   # 3072
CHAT_MODEL = "gpt-4o-mini"

# ---------- QDRANT ----------
QDRANT_URL = "https://qdrant.dev.adapstory.com"
COLLECTION_NAME = "presentations_industrix_openai"
TOP_K = 10

os.environ["QDRANT_DISABLE_CHECK"] = "1"

qdrant = QdrantClient(
    url=QDRANT_URL,
    port=443,
    timeout=120
)


In [4]:
import numpy as np

def embed_query(text: str) -> list[float]:
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return response.data[0].embedding


In [5]:
def retrieve_context(question, top_k=TOP_K):
    vector = embed_query(question)

    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=vector,
        limit=top_k,
        with_payload=True
    )

    contexts = []

    for r in results:
        payload = r.payload or {}
        text = payload.get("text", "").strip()
        if not text:
            continue

        contexts.append({
            "score": r.score,
            "text": text,
            "page": payload.get("page"),
            "source": payload.get("source")
        })

    return contexts


In [6]:
def debug_retrieval(question):
    contexts = retrieve_context(question)

    print(f"\n🔍 Вопрос: {question}\n")
    for i, c in enumerate(contexts, 1):
       # print(f"--- TOP {i} ---")
        #print(f"Score: {c['score']:.3f}")
        #print(f"Source: {c['source']} | Page: {c['page']}")
        print(c["text"][:500])
        print()


In [7]:
# tutor_engine.py

#from rag_core import retrieve_context, ask_openai

def build_tutor_prompt(contexts, question, history=None):
    context_text = "\n\n".join(
        f"[стр. {c['page']}]\n{c['text']}"
        for c in contexts
    )

    history_text = ""
    if history:
        history_text = "\n".join(history)

    return f"""
Ты — ИИ-тьютор компании Industrix.

Твоя задача — НЕ давать готовый ответ.
Твоя задача — помочь студенту прийти к ответу самостоятельно.

Правила:
1. Задай ОДИН наводящий вопрос.
2. Не раскрывай полный ответ.
3. Опирайся только на контекст ниже.
4. Если информации нет — скажи: 
   "В материалах нет информации для разбора этого вопроса."

КОНТЕКСТ:
{context_text}

ИСТОРИЯ ДИАЛОГА:
{history_text}

ВОПРОС СТУДЕНТА:
{question}

ТВОЙ НАВОДЯЩИЙ ВОПРОС:
"""

In [ ]:

def build_prompt(contexts, question):
    context_text = "\n\n".join(
        f"[стр. {c['page']} | score {c['score']:.2f}]\n{c['text']}"
        for c in contexts
    )

    return f"""
Ты — эксперт по материалам компании Industrix.
Отвечай СТРОГО на основе контекста ниже.
Если ответа в контексте нет — скажи:
"В предоставленных материалах нет информации".

КОНТЕКСТ:
{context_text}

ВОПРОС:
{question}

ОТВЕТ:
"""


In [8]:
def ask_openai(prompt):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=800
    )
    return response.choices[0].message.content


In [9]:
def tutor_answer(question, history=None):
    contexts = retrieve_context(question)

    if not contexts:
        return "В базе нет релевантной информации."

    prompt = build_tutor_prompt(contexts, question, history)

    return ask_openai(prompt)

In [11]:
def build_evaluation_prompt(contexts, question, student_answer):
    context_text = "\n\n".join(
        c["text"] for c in contexts
    )

    return f"""
Ты проверяешь понимание студента.

КОНТЕКСТ (правильная информация):
{context_text}

ВОПРОС:
{question}

ОТВЕТ СТУДЕНТА:
{student_answer}

1. Верно ли понимание?
2. Если есть ошибка — кратко укажи направление.
3. Задай следующий наводящий вопрос.
4. Не раскрывай полный ответ.

ОТВЕТ:
"""

In [12]:
def evaluate_student(question, student_answer):
    contexts = retrieve_context(question)

    if not contexts:
        return "В материалах нет информации для оценки."

    prompt = build_evaluation_prompt(contexts, question, student_answer)

    return ask_openai(prompt)

In [14]:
question = "Какие причины провала стартапа?"

# Шаг 1 — задаём наводящий вопрос
tutor_msg = tutor_answer(question)
print("Тьютор:", tutor_msg)

# Студент отвечает
student_reply = input("Студент: ")

# Шаг 2 — оцениваем
feedback = evaluate_student(question, student_reply)
print("Тьютор:", feedback)

Тьютор: Как вы думаете, какая из причин, упомянутых в материалах, может быть наиболее критичной для успеха стартапа?
Тьютор: 1. Нет, понимание неверное. В контексте были указаны конкретные причины провала стартапов.
2. Основные причины включают отсутствие рыночной потребности, нехватку денег, неправильную команду, конкуренцию и проблемы с ценообразованием.
3. Какие, по твоему мнению, факторы могут влиять на успешность стартапа?


In [10]:
# ---------- Детектор незнания ----------

def is_no_knowledge_answer(text: str) -> bool:
    triggers = [
        "не знаю",
        "не читал",
        "без понятия",
        "затрудняюсь",
        "не уверен",
        "не помню"
    ]
    text = text.lower()
    return any(t in text for t in triggers)


# ---------- Prompt: сократический режим ----------

def build_socratic_prompt(contexts, question, history):
    context_text = "\n\n".join(
        f"[Источник: {c['source']} | стр. {c['page']}]\n{c['text']}"
        for c in contexts[:3]
    )

    history_text = "\n".join(history)

    return f"""
Ты — ИИ-тьютор компании Industrix.

Твоя задача — НЕ давать готовый ответ.
Задай один наводящий вопрос, который поможет студенту самому прийти к ответу.

Опирайся только на контекст ниже.

КОНТЕКСТ:
{context_text}

ИСТОРИЯ:
{history_text}

ВОПРОС:
{question}

НАВОДЯЩИЙ ВОПРОС:
"""


# ---------- Prompt: объяснение ----------

def build_explanation_prompt(contexts, question):
    context_text = "\n\n".join(
        f"[Источник: {c['source']} | стр. {c['page']}]\n{c['text']}"
        for c in contexts[:3]
    )

    return f"""
Ты — ИИ-тьютор компании Industrix.

Студент сообщил, что не знает ответ.

Объясни тему кратко и понятно.
После объяснения обязательно укажи источник (название файла и страницу).

КОНТЕКСТ:
{context_text}

ВОПРОС:
{question}

ОБЪЯСНЕНИЕ:
"""


# ---------- Prompt: оценка ----------

def build_evaluation_prompt(contexts, question, student_answer):
    context_text = "\n\n".join(
        c["text"] for c in contexts[:3]
    )

    return f"""
Ты проверяешь понимание студента.

КОНТЕКСТ:
{context_text}

ВОПРОС:
{question}

ОТВЕТ СТУДЕНТА:
{student_answer}

1. Кратко оцени, верно ли понимание.
2. Если частично верно — мягко поправь.
3. Если верно — похвали и задай усложняющий вопрос.
4. Не раскрывай полный ответ.

ОТВЕТ:
"""


# ---------- Основной класс тьютора ----------

class TutorSession:

    def __init__(self, question):
        self.question = question
        self.contexts = retrieve_context(question)
        self.history = []
        self.attempts = 0
        self.mastered = False

    def start(self):
        if not self.contexts:
            print("В базе нет информации по этому вопросу.")
            return

        print("\n🧠 Начинаем обучение по теме:\n")

        tutor_msg = ask_openai(
            build_socratic_prompt(self.contexts, self.question, self.history)
        )

        print("Тьютор:", tutor_msg)

        while not self.mastered and self.attempts < 5:

            student_reply = input("\nСтудент: ")

            self.history.append(f"Тьютор: {tutor_msg}")
            self.history.append(f"Студент: {student_reply}")

            # если студент не знает → объяснение
            if is_no_knowledge_answer(student_reply):
                explanation = ask_openai(
                    build_explanation_prompt(self.contexts, self.question)
                )
                print("\nТьютор:", explanation)
                self.mastered = True
                break

            # иначе оцениваем
            feedback = ask_openai(
                build_evaluation_prompt(
                    self.contexts,
                    self.question,
                    student_reply
                )
            )

            print("\nТьютор:", feedback)

            if "верно" in feedback.lower() and "услож" not in feedback.lower():
                self.mastered = True
                break

            tutor_msg = feedback
            self.attempts += 1

        print("\n✅ Сессия завершена.\n")

In [13]:
question = input("Введите тему или вопрос: ")

session = TutorSession(question)
session.start()

APITimeoutError: Request timed out.

In [14]:
from enum import Enum


class TutorMode(str, Enum):
    SOCRATIC = "socratic"
    HINT = "hint"
    EXPLAIN = "explain"

In [15]:
def build_tutor_prompt(contexts, question, history, mode: TutorMode):

    context_text = "\n\n".join(
        f"[Источник: {c['source']} | стр. {c['page']}]\n{c['text']}"
        for c in contexts[:3]
    )

    history_text = "\n".join(history)

    mode_instruction = {
        TutorMode.SOCRATIC: """
Задай ОДИН наводящий вопрос.
Не раскрывай ответ.
""",

        TutorMode.HINT: """
Дай небольшую подсказку (1–3 предложения).
Не раскрывай полный ответ.
""",

        TutorMode.EXPLAIN: """
Кратко объясни тему (4–6 предложений).
После объяснения укажи источник (файл и страницу).
"""
    }[mode]

    return f"""
Ты — ИИ-тьютор компании Industrix.

{mode_instruction}

Опирайся только на контекст ниже.

КОНТЕКСТ:
{context_text}

ИСТОРИЯ:
{history_text}

ВОПРОС:
{question}

ОТВЕТ:
"""

In [16]:
def tutor_step(question, history, mode: TutorMode):
    contexts = retrieve_context(question)

    if not contexts:
        return "В базе нет релевантной информации."

    prompt = build_tutor_prompt(contexts, question, history, mode)

    return ask_openai(prompt)

In [17]:
def choose_mode(student_answer, attempts):

    if is_no_knowledge_answer(student_answer):
        return TutorMode.EXPLAIN

    if attempts == 0:
        return TutorMode.SOCRATIC

    if attempts == 1:
        return TutorMode.HINT

    if attempts >= 2:
        return TutorMode.EXPLAIN

    return TutorMode.SOCRATIC

In [18]:
class TutorSession:

    def __init__(self, question):
        self.question = question
        self.contexts = retrieve_context(question)
        self.history = []
        self.attempts = 0
        self.mastered = False

    def start(self):
        if not self.contexts:
            print("Нет информации по теме.")
            return

        mode = TutorMode.SOCRATIC

        while not self.mastered and self.attempts < 5:

            tutor_msg = tutor_step(
                self.question,
                self.history,
                mode
            )

            print("\nТьютор:", tutor_msg)

            student_reply = input("\nСтудент: ")

            self.history.append(f"Тьютор: {tutor_msg}")
            self.history.append(f"Студент: {student_reply}")

            mode = choose_mode(student_reply, self.attempts)

            if mode == TutorMode.EXPLAIN:
                tutor_msg = tutor_step(
                    self.question,
                    self.history,
                    TutorMode.EXPLAIN
                )
                print("\nТьютор:", tutor_msg)
                self.mastered = True
                break

            self.attempts += 1

        print("\nСессия завершена.")

In [22]:

question = input("Введите тему или вопрос: ")

session = TutorSession(question)
session.start()


Тьютор: Какие факторы, связанные с командой и конкурентами, вы считаете наиболее критичными для успеха вашего стартапа в области edtech?

Тьютор: Какие аспекты вашего продукта вы считаете наиболее важными для привлечения интереса бизнес-заказчиков?

Тьютор: Обратите внимание на состав вашей команды и её компетенции, а также на конкурентное окружение. Также важно, чтобы ваше предложение действительно решало проблемы бизнес-заказчиков и приносило им экономическую выгоду.

Тьютор: Причины, способствующие провалу стартапов в области edtech, могут быть разнообразными. Во-первых, наличие неправильной команды, которая не обладает необходимыми компетенциями, может существенно снизить шансы на успех. Во-вторых, конкуренция на рынке может оказаться слишком сильной, что приведет к вытеснению вашего продукта. Также важным аспектом является ценообразование: если ваше предложение не создает должного экономического эффекта для бизнес-заказчиков или не решает их реальные проблемы, это может стать при

ИИ агент-тьютор

In [4]:
import json
from dataclasses import dataclass, field
from typing import List

from openai import OpenAI
from qdrant_client import QdrantClient

In [ ]:
# ---------- OPENAI ----------
client = OpenAI(api_key="")  # берет ключ из ENV

EMBED_MODEL = "text-embedding-3-large"   # 3072
CHAT_MODEL = "gpt-4o-mini"

# ---------- QDRANT ----------
QDRANT_URL = "https://qdrant.dev.adapstory.com"
COLLECTION_NAME = "presentations_industrix_openai"
TOP_K = 10

os.environ["QDRANT_DISABLE_CHECK"] = "1"

qdrant = QdrantClient(
    url=QDRANT_URL,
    port=443,
    timeout=120
)

In [6]:
def embed_query(text):
    
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    
    return response.data[0].embedding


def ask_llm(prompt, temperature=0.3):

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=700
    )

    return response.choices[0].message.content.strip()

In [7]:
def retrieve_context(question, top_k=5):

    vector = embed_query(question)

    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=vector,
        limit=top_k,
        with_payload=True
    )

    contexts = []

    for r in results:

        payload = r.payload or {}

        text = payload.get("text", "").strip()

        if not text:
            continue

        contexts.append({
            "text": text,
            "page": payload.get("page"),
            "source": payload.get("source"),
            "score": r.score
        })

    return contexts

In [17]:
def build_context(contexts):

    formatted_chunks = []

    for i, c in enumerate(contexts):

        formatted_chunks.append(
            f"""
CHUNK_ID: {i}

SOURCE: {c.get("source")}
PAGE: {c.get("page")}

TEXT:
{c.get("text")}
"""
        )

    return "\n\n".join(formatted_chunks)

In [18]:
import json
from dataclasses import dataclass, field
from typing import List

# --- Student State ---
@dataclass
class StudentState:
    topic: str
    attempts: int = 0
    mastery_score: float = 0.0
    history: List[str] = field(default_factory=list)
    misconceptions: List[str] = field(default_factory=list)
    last_strategy: str | None = None
    explanation_used: bool = False

In [19]:
class PlannerAgent:

    def plan(self, question, context_text, student_state, last_answer=None, evaluation=None):

        prompt = f"""
Ты педагогический планировщик ИИ-тьютора.

Выбери стратегию обучения: 
- socratic → задать наводящий вопрос
- hint → дать подсказку
- explain → объяснить тему

Правила:
1. Если студент ответил правильно или score > 0.8 → не нужно socratic
2. Если студент явно не понял тему → hint
3. Если attempts >= 2 или объяснение уже требуется → explain
4. Не повторяй socratic бесконечно

Текущее состояние:
attempts: {student_state.attempts}
last_strategy: {student_state.last_strategy}
explanation_used: {student_state.explanation_used}

Последний ответ студента:
{last_answer or "пока нет ответа"}

Evaluation последнего ответа:
{evaluation or "нет"}

Контекст:
{context_text}

Верни JSON: {{"strategy": "socratic|hint|explain"}}
"""

        response = ask_llm(prompt, temperature=0.2)

        try:
            plan = json.loads(response)
            strategy = plan.get("strategy", "socratic")
        except:
            strategy = "socratic"

        return {"strategy": strategy}

In [20]:
class TutorAgent:

    def respond(self, strategy, question, contexts, history):

        context_text = build_context(contexts)

        prompt = f"""
Ты ИИ-тьютор курса.

Ты обучаешь студента по материалам курса.
Используй ТОЛЬКО информацию из контекста.

ЗАПРЕЩЕНО:
- использовать знания вне контекста
- придумывать источники
- ссылаться на исследования или сайты

Если информации нет — скажи:
"В материалах курса нет информации".

Контекст материалов курса:

{context_text}

Вопрос студента:
{question}

История диалога:
{history}

Стратегия обучения: {strategy}

Правила стратегий:

socratic:
- задай один наводящий вопрос
- не давай ответ
- помоги студенту подумать

hint:
- дай маленькую подсказку
- НЕ раскрывай полный ответ
- можно намекнуть на направление мысли
- максимум 2 предложения

explain:
- объясни ответ кратко
- используй информацию из контекста
- добавь источник в формате:

[Источник: SOURCE, стр. PAGE]

Ответ:
"""

        return ask_llm(prompt, temperature=0.3)

In [23]:
class EvaluatorAgent:

    def evaluate(self, question, student_answer, context):

        prompt = f"""
Ты оцениваешь ответ студента.

Вопрос:
{question}

Ответ студента:
{student_answer}

Контекст курса:
{context}

Верни JSON:

{{
 "correct": true/false,
 "partial": true/false,
 "misconception": "если есть",
 "score": число от 0 до 1
}}
"""

        response = ask_llm(prompt, temperature=0)

        try:
            return json.loads(response)
        except:
            return {
                "correct": False,
                "partial": False,
                "misconception": "",
                "score": 0.0
            }

In [24]:
class TutorOrchestrator:

    def __init__(self, question):

        self.question = question

        # RAG retrieval
        self.contexts = retrieve_context(question)

        self.context_text = "\n\n".join(
            c["text"] for c in self.contexts[:3]
        )

        # состояние студента
        self.state = StudentState(topic=question)

        # агенты
        self.planner = PlannerAgent()
        self.tutor = TutorAgent()
        self.evaluator = EvaluatorAgent()

    def run(self):

        last_answer = None
        evaluation = None

        if not self.contexts:
            print("Нет информации в базе.")
            return

        print("\n📚 Начинаем обучение\n")

        while True:

            # --- 1. Планируем стратегию ---
            plan = self.planner.plan(
                self.question,
                self.context_text,
                self.state,
                last_answer,
                evaluation
            )
            strategy = plan["strategy"]

            print(f"\n📊 Strategy: {strategy}")

            # --- 2. Генерируем ответ тьютора ---
            tutor_msg = self.tutor.respond(
                strategy,
                self.question,
                self.contexts,
                self.state.history
            )
            print("\n🧠 Тьютор:", tutor_msg)

            self.state.last_strategy = strategy
            self.state.history.append(f"Тьютор: {tutor_msg}")

            # --- 3. explain завершает сессию ---
            if strategy == "explain":
                self.state.explanation_used = True
                print("\n📚 Тема объяснена. Сессия завершена.")
                break

            # --- 4. Ввод студента ---
            last_answer = input("\n👨‍🎓 Студент: ")

            self.state.attempts += 1

            # --- 5. Оценка ответа ---
            evaluation = self.evaluator.evaluate(
                self.question,
                last_answer,
                self.context_text
            )
            print("\n📊 Evaluation:", evaluation)

            # --- 6. Обновляем состояние ---
            self.state.mastery_score = evaluation.get("score", 0.0)

            if evaluation.get("misconception"):
                self.state.misconceptions.append(evaluation["misconception"])

            self.state.history.append(f"Студент: {last_answer}")

            # --- 7. Проверка усвоения темы ---
            if evaluation.get("correct") or evaluation.get("score",0) > 0.8:
                print("\n✅ Отлично! Тема усвоена.")
                break

            # --- 8. Защита от бесконечного цикла ---
            if self.state.attempts >= 4:
                print("\n⚠️ Слишком много попыток, переходим к объяснению темы.")
                tutor_msg = self.tutor.respond(
                    "explain",
                    self.question,
                    self.context_text,
                    self.state.history
                )
                print("\n🧠 Тьютор:", tutor_msg)
                break

In [25]:
question = "Какие причины провала стартапа?"

session = TutorOrchestrator(question)

session.run()


📚 Начинаем обучение


📊 Strategy: socratic

🧠 Тьютор: Какие факторы, по твоему мнению, могут привести к тому, что стартап не сможет успешно развиваться на рынке? Подумай о том, что может повлиять на его жизнеспособность.

📊 Evaluation: {'correct': False, 'partial': True, 'misconception': 'Ответ не включает все ключевые причины провала стартапов, такие как неправильное ценообразование и недостаток интереса со стороны рынка.', 'score': 0.5}

📊 Strategy: hint

🧠 Тьютор: Подумай о том, какие еще факторы могут повлиять на успешность стартапа, кроме команды и финансов. Например, как важны рыночные потребности и конкуренция?

📊 Evaluation: {'correct': False, 'partial': True, 'misconception': 'Ответ студента не охватывает все основные причины провала стартапов, такие как неправильная команда, конкуренция и проблемы с ценообразованием.', 'score': 0.4}

📊 Strategy: explain

🧠 Тьютор: Основные причины провала стартапов включают следующие факторы:

1. Отсутствие рыночной потребности.
2. Закончили

In [20]:
pip install -q python-telegram-bot==20.7 

Note: you may need to restart the kernel to use updated packages.


In [21]:
pip install -q nest_asyncio


Note: you may need to restart the kernel to use updated packages.


In [22]:
def rag_answer(question: str) -> str:
    contexts = retrieve_context(question)  # ← list

    if not contexts:
        return "Я не нашёл информацию в документах."

    context_text = "\n\n".join(
        c["text"] for c in contexts
    )

    prompt = f"""
Ты отвечаешь ТОЛЬКО на основе контекста ниже.

Контекст:
{context_text}

Вопрос:
{question}

Ответ:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content.strip()


In [23]:
from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters,
)
import os


In [24]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Привет! Задай вопрос по материалам Industrix 📄"
    )


In [25]:
import asyncio

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    question = update.message.text

    await update.message.reply_text("⏳ Думаю...")

    loop = asyncio.get_running_loop()
    answer = await loop.run_in_executor(
        None,
        rag_answer,
        question
    )

    await update.message.reply_text(answer[:4000])  # лимит TG


In [ ]:
BOT_TOKEN = ""

import asyncio

async def main():
    app = ApplicationBuilder().token(BOT_TOKEN).build()

    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

    print("🚀 Telegram RAG bot started")

    await app.initialize()
    await app.start()
    await app.bot.initialize()
    await app.updater.start_polling()

# В Jupyter:
await main()



🚀 Telegram RAG bot started


Error while getting Updates: httpx.ReadError: 
Exception happened while polling for updates.
Traceback (most recent call last):
  File "c:\Users\MSI\anaconda3\Lib\site-packages\httpcore\_exceptions.py", line 10, in map_exceptions
    yield
  File "c:\Users\MSI\anaconda3\Lib\site-packages\httpcore\_backends\anyio.py", line 34, in read
    return await self._stream.receive(max_bytes=max_bytes)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\anyio\streams\tls.py", line 205, in receive
    data = await self._call_sslobject_method(self._ssl_object.read, max_bytes)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\anyio\streams\tls.py", line 147, in _call_sslobject_method
    data = await self.transport_stream.receive()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\anyio\_backends\_asyncio.py", line 1132,

In [ ]:
from telegram import Bot

BOT_TOKEN = ""

bot = Bot(BOT_TOKEN)

await bot.delete_webhook(drop_pending_updates=True)

print("✅ Webhook и очередь обновлений сброшены")


✅ Webhook и очередь обновлений сброшены


Error while getting Updates: Conflict: terminated by setWebhook request
Exception happened while polling for updates.
Traceback (most recent call last):
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\ext\_updater.py", line 688, in _network_loop_retry
    if not await action_cb():
           ^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\ext\_updater.py", line 384, in polling_action_cb
    raise exc
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\ext\_updater.py", line 373, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\ext\_extbot.py", line 558, in get_updates
    updates = await super().get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSI\anaconda3\Lib\site-packages\telegram\_bot.py", line 525, in decorator
    result = await func(self, *args, **kwargs)  # skipcq: PYL-E1102
             ^^^^^^^^^^^^^

ИИ-тьютор доработка

In [ ]:
import os
import json
from dataclasses import dataclass, field
from typing import List, Optional

from openai import OpenAI
from qdrant_client import QdrantClient

client = OpenAI(api_key="")  # берет ключ из ENV

EMBED_MODEL = "text-embedding-3-large"
CHAT_MODEL = "gpt-4o-mini"

QDRANT_URL = "https://qdrant.dev.adapstory.com"
COLLECTION_NAME = "presentations_industrix_openai"

os.environ["QDRANT_DISABLE_CHECK"] = "1"

qdrant = QdrantClient(url=QDRANT_URL, port=443, timeout=120)


# ─────────────────────────── helpers ────────────────────────────

def embed_query(text: str) -> list:
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding


def ask_llm(prompt: str, temperature: float = 0.3) -> str:
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=700,
    )
    return response.choices[0].message.content.strip()


def retrieve_context(question: str, top_k: int = 10) -> list:
    vector = embed_query(question)
    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=vector,
        limit=top_k,
        with_payload=True,
    )
    contexts = []
    for r in results:
        payload = r.payload or {}
        text = payload.get("text", "").strip()
        if not text:
            continue
        contexts.append({
            "text": text,
            "page": payload.get("page"),
            "source": payload.get("source"),
            "score": r.score,
        })
    return contexts


def build_context(contexts: list) -> str:
    chunks = []
    for i, c in enumerate(contexts):
        chunks.append(
            f"CHUNK_ID: {i}\nSOURCE: {c.get('source')}\nPAGE: {c.get('page')}\n\nTEXT:\n{c.get('text')}"
        )
    return "\n\n---\n\n".join(chunks)


def format_history(history: list) -> str:
    """Форматирует историю диалога в читаемую строку."""
    return "\n".join(history) if history else "История диалога пуста."


# ─────────────────────────── state ──────────────────────────────

@dataclass
class StudentState:
    topic: str
    attempts: int = 0
    mastery_score: float = 0.0
    history: List[str] = field(default_factory=list)
    misconceptions: List[str] = field(default_factory=list)
    last_strategy: Optional[str] = None
    explanation_used: bool = False
    consecutive_wrong: int = 0   # ← счётчик подряд идущих неверных ответов


# ─────────────────────────── agents ─────────────────────────────

class PlannerAgent:
    """
    Стратегии:
      socratic  — задаём наводящий вопрос
      hint      — даём подсказку
      explain   — объясняем тему
      verify    — после объяснения проверяем понимание (контрольный вопрос)
    """

    def plan(
        self,
        question: str,
        context_text: str,
        state: StudentState,
        last_answer: Optional[str] = None,
        evaluation: Optional[dict] = None,
    ) -> dict:

        misconceptions_text = (
            "\n".join(f"- {m}" for m in state.misconceptions)
            if state.misconceptions
            else "нет"
        )

        prompt = f"""
Ты педагогический планировщик ИИ-тьютора.

Выбери ОДНУ стратегию из: socratic, hint, explain, verify.

Правила выбора:
1. socratic  — студент ещё не пробовал отвечать или ответил частично (score < 0.5), attempts <= 1
2. hint      — студент ответил, но score < 0.6, attempts <= 3; или есть явное заблуждение
3. explain   — attempts >= 3 ИЛИ consecutive_wrong >= 2 ИЛИ студент явно не понимает
4. verify    — стратегия была explain → нужно проверить понимание (задать короткий вопрос)
5. НЕ повторяй одну стратегию больше 2 раз подряд без изменений
6. Если score > 0.8 → ответь "mastered" (тема усвоена, сессию можно завершать)

Текущее состояние:
- attempts: {state.attempts}
- consecutive_wrong: {state.consecutive_wrong}
- last_strategy: {state.last_strategy or "нет"}
- explanation_used: {state.explanation_used}
- mastery_score: {state.mastery_score}

Заблуждения студента:
{misconceptions_text}

Последний ответ студента:
{last_answer or "пока нет ответа"}

Оценка последнего ответа:
{json.dumps(evaluation, ensure_ascii=False) if evaluation else "нет"}

Контекст (фрагмент):
{context_text[:800]}

Верни ТОЛЬКО JSON без пояснений:
{{"strategy": "socratic|hint|explain|verify|mastered"}}
"""

        response = ask_llm(prompt, temperature=0.1)

        try:
            # убираем возможные markdown-обёртки
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            plan = json.loads(cleaned)
            strategy = plan.get("strategy", "socratic")
        except Exception:
            strategy = "socratic"

        return {"strategy": strategy}


class TutorAgent:

    def respond(
        self,
        strategy: str,
        question: str,
        contexts: list,
        history: list,
        misconceptions: Optional[List[str]] = None,
    ) -> str:

        context_text = build_context(contexts)
        history_text = format_history(history)
        misc_text = (
            "Заблуждения студента, которые нужно исправить:\n"
            + "\n".join(f"- {m}" for m in misconceptions)
            if misconceptions
            else ""
        )

        strategy_instructions = {
            "socratic": (
                "Задай ОДИН наводящий вопрос, который помогает студенту самому прийти к ответу.\n"
                "Не давай ответ и не перечисляй факты. Максимум — 2 предложения."
            ),
            "hint": (
                "Дай ОДНУ небольшую подсказку, указывающую направление мысли.\n"
                "Не раскрывай полный ответ. Упомяни заблуждение студента, если оно есть.\n"
                "Максимум — 3 предложения."
            ),
            "explain": (
                "Объясни тему развёрнуто и структурированно.\n"
                "Используй ТОЛЬКО контекст курса.\n"
                "В конце укажи источник: [Источник: SOURCE, стр. PAGE]\n"
                "Исправь заблуждения студента, если они есть."
            ),
            "verify": (
                "Ты только что объяснил тему. Теперь задай студенту ОДИН короткий проверочный вопрос,\n"
                "чтобы убедиться, что он понял объяснение. Вопрос должен быть конкретным."
            ),
        }

        instruction = strategy_instructions.get(strategy, strategy_instructions["socratic"])

        prompt = f"""
Ты ИИ-тьютор курса. Используй ТОЛЬКО информацию из контекста.
Если информации нет — скажи: «В материалах курса нет информации по этому вопросу».

КОНТЕКСТ КУРСА:
{context_text}

ВОПРОС СТУДЕНТА:
{question}

ИСТОРИЯ ДИАЛОГА:
{history_text}

{misc_text}

СТРАТЕГИЯ: {strategy}
ИНСТРУКЦИЯ:
{instruction}

ОТВЕТ ТЬЮТОРА:
"""

        return ask_llm(prompt, temperature=0.3)


class IntentClassifier:
    """
    Определяет намерение сообщения студента:
      answer        — попытка ответить на вопрос тьютора
      chat          — разговорное сообщение (спасибо, понял, окей и т.д.)
      question      — студент задаёт свой вопрос по теме
      off_topic     — сообщение не по теме
    """

    def classify(self, student_message: str, tutor_last_message: str, topic: str) -> str:

        prompt = f"""
Ты классификатор намерений в диалоге тьютора и студента.

Тема обучения: {topic}

Последнее сообщение тьютора:
{tutor_last_message}

Сообщение студента:
{student_message}

Определи намерение сообщения студента. Варианты:
- answer     — студент пытается ответить на вопрос тьютора (даже частично или неверно)
- chat       — разговорное сообщение: благодарность, подтверждение, "понял", "окей", "спасибо" и т.п.
- question   — студент задаёт свой вопрос по теме курса
- off_topic  — сообщение не связано с темой обучения

Верни ТОЛЬКО JSON:
{{"intent": "answer|chat|question|off_topic"}}
"""

        response = ask_llm(prompt, temperature=0)

        try:
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            return json.loads(cleaned).get("intent", "answer")
        except Exception:
            return "answer"


class ChatAgent:
    """
    Отвечает на разговорные сообщения и вопросы студента,
    после чего мягко возвращает к теме обучения.
    """

    def respond_to_chat(
        self,
        student_message: str,
        tutor_last_message: str,
        topic: str,
        history: list,
    ) -> str:

        history_text = format_history(history[-6:])  # последние 6 реплик

        prompt = f"""
Ты ИИ-тьютор курса. Студент написал тебе разговорное сообщение.

Тема обучения: {topic}

История диалога:
{history_text}

Твоё последнее сообщение:
{tutor_last_message}

Сообщение студента:
{student_message}

Ответь естественно и коротко (1-2 предложения).
Затем мягко напомни свой предыдущий вопрос или предложи продолжить.
Не повторяй весь вопрос дословно — перефразируй кратко.
"""

        return ask_llm(prompt, temperature=0.4)

    def respond_to_question(
        self,
        student_question: str,
        contexts: list,
        topic: str,
        history: list,
        tutor_last_message: str,
    ) -> str:

        context_text = build_context(contexts[:3])
        history_text = format_history(history[-6:])

        prompt = f"""
Ты ИИ-тьютор курса. Студент задал уточняющий вопрос.

Тема обучения: {topic}
Контекст курса: {context_text}

История диалога:
{history_text}

Вопрос студента: {student_question}

Ответь кратко, используя только контекст курса (2-4 предложения).
Если информации нет — скажи честно.
После ответа мягко верни студента к теме, напомнив свой вопрос одним предложением.

Твой предыдущий вопрос был: {tutor_last_message}
"""

        return ask_llm(prompt, temperature=0.3)


class EvaluatorAgent:

    def evaluate(self, question: str, student_answer: str, context: str) -> dict:

        prompt = f"""
Ты оцениваешь ответ студента на вопрос по материалам курса.

Вопрос: {question}
Ответ студента: {student_answer}
Контекст курса: {context}

Верни ТОЛЬКО JSON без пояснений:
{{
  "correct": true/false,
  "partial": true/false,
  "misconception": "описание заблуждения или пустая строка",
  "score": 0.0-1.0,
  "feedback": "краткий комментарий для тьютора (не для студента)"
}}
"""

        response = ask_llm(prompt, temperature=0)

        try:
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            return json.loads(cleaned)
        except Exception:
            return {"correct": False, "partial": False, "misconception": "", "score": 0.0, "feedback": ""}


# ─────────────────────────── orchestrator ───────────────────────

class TutorOrchestrator:

    MAX_ATTEMPTS = 6  # максимум попыток-ответов (chat/question не считаются)

    def __init__(self, question: str):
        self.question = question
        self.contexts = retrieve_context(question)
        self.context_text = build_context(self.contexts[:5])

        self.state = StudentState(topic=question)

        self.planner = PlannerAgent()
        self.tutor = TutorAgent()
        self.evaluator = EvaluatorAgent()
        self.intent_clf = IntentClassifier()
        self.chat_agent = ChatAgent()

        self._last_tutor_msg = ""  # последнее сообщение тьютора для контекста

    # ── helpers ──────────────────────────────────────────────────

    def _update_state_after_eval(self, evaluation: dict, last_answer: str):
        score = evaluation.get("score", 0.0)
        self.state.mastery_score = score
        self.state.attempts += 1

        misconception = evaluation.get("misconception", "")
        if misconception and misconception not in self.state.misconceptions:
            self.state.misconceptions.append(misconception)

        if not evaluation.get("correct") and score < 0.5:
            self.state.consecutive_wrong += 1
        else:
            self.state.consecutive_wrong = 0

        self.state.history.append(f"Студент: {last_answer}")

    def _print_sep(self):
        print("\n" + "─" * 50)

    # ── main loop ────────────────────────────────────────────────

    def run(self):
        if not self.contexts:
            print("❌ Нет информации в базе знаний по данному вопросу.")
            return

        print("\n📚 Начинаем обучение\n")
        print(f"📌 Тема: {self.question}\n")

        last_answer = None
        evaluation = None
        force_next_strategy = None  # гарантированная стратегия для следующего хода

        while True:

            # ════════════════════════════════════════════
            # ФАЗА 1: ТЬЮТОР ГОВОРИТ
            # ════════════════════════════════════════════

            # ── 1. Определяем стратегию ─────────────────
            if force_next_strategy:
                # Детерминированный переход (например, explain → verify)
                strategy = force_next_strategy
                force_next_strategy = None
            else:
                plan = self.planner.plan(
                    self.question,
                    self.context_text,
                    self.state,
                    last_answer,
                    evaluation,
                )
                strategy = plan["strategy"]

            # ── 2. Тема усвоена → выход ──────────────────
            if strategy == "mastered":
                print("\n✅ Отлично! Тьютор подтверждает: тема усвоена.")
                break

            # ── 3. Лимит попыток → выход ─────────────────
            if self.state.attempts >= self.MAX_ATTEMPTS:
                print("\n⚠️  Достигнут лимит попыток. Завершаем сессию.")
                break

            self._print_sep()
            print(f"📊 Стратегия: {strategy}")

            # ── 4. Тьютор отвечает ───────────────────────
            tutor_msg = self.tutor.respond(
                strategy,
                self.question,
                self.contexts,
                self.state.history,
                self.state.misconceptions,
            )
            print(f"\n🧠 Тьютор: {tutor_msg}")

            self.state.last_strategy = strategy
            self.state.history.append(f"Тьютор: {tutor_msg}")
            self._last_tutor_msg = tutor_msg

            # После explain → следующий ход ВСЕГДА verify (не доверяем LLM)
            if strategy == "explain":
                self.state.explanation_used = True
                force_next_strategy = "verify"

            # ════════════════════════════════════════════
            # ФАЗА 2: СТУДЕНТ ГОВОРИТ
            # (цикл внутри фазы — только для chat/question/off_topic)
            # ════════════════════════════════════════════

            while True:
                print()
                raw_input = input("👨‍🎓 Студент: ").strip()

                if not raw_input:
                    print("(тьютор ждёт ответа...)")
                    continue

                # ── 5. Классифицируем намерение ──────────
                intent = self.intent_clf.classify(
                    raw_input,
                    self._last_tutor_msg,
                    self.question,
                )
                print(f"💬 Намерение: {intent}")

                # ── 6. chat → отвечаем, ждём дальше ──────
                if intent == "chat":
                    reply = self.chat_agent.respond_to_chat(
                        raw_input,
                        self._last_tutor_msg,
                        self.question,
                        self.state.history,
                    )
                    print(f"\n🧠 Тьютор: {reply}")

                    # ✅ НОВОЕ УСЛОВИЕ ЗАВЕРШЕНИЯ
                    if strategy in ["verify", "explain"]:
                        print("\n✅ Сессия завершена. Удачи в обучении!")
                        return

                    self.state.history.append(f"Студент: {raw_input}")
                    self.state.history.append(f"Тьютор: {reply}")
                    self._last_tutor_msg = reply

                    continue

                # ── 7. question → отвечаем, ждём дальше ──
                if intent == "question":
                    reply = self.chat_agent.respond_to_question(
                        raw_input,
                        self.contexts,
                        self.question,
                        self.state.history,
                        self._last_tutor_msg,
                    )
                    print(f"\n🧠 Тьютор: {reply}")
                    self.state.history.append(f"Студент: {raw_input}")
                    self.state.history.append(f"Тьютор: {reply}")
                    self._last_tutor_msg = reply
                    continue

                # ── 8. off_topic → напоминаем, ждём дальше
                if intent == "off_topic":
                    print(f"\n🧠 Тьютор: Давай вернёмся к нашей теме. {self._last_tutor_msg}")
                    self.state.history.append(f"Студент: {raw_input}")
                    continue

                # ── 9. answer → выходим из фазы 2 ────────
                break  # переходим к оценке

            # ════════════════════════════════════════════
            # ФАЗА 3: ОЦЕНИВАЕМ ОТВЕТ
            # ════════════════════════════════════════════

            last_answer = raw_input

            evaluation = self.evaluator.evaluate(
                self.question,
                last_answer,
                self.context_text,
            )

            score = evaluation.get("score", 0.0)
            feedback = evaluation.get("feedback", "")
            print(f"\n📊 Оценка: score={score:.2f} | {feedback}")

            self._update_state_after_eval(evaluation, last_answer)

            # ── 10. Проверяем усвоение ───────────────────
            if evaluation.get("correct") or score >= 0.85:
                print("\n✅ Правильно! Тема усвоена.")
                break


# ─────────────────────────── entry point ────────────────────────

if __name__ == "__main__":
    question = "Какие причины провала стартапа?"
    session = TutorOrchestrator(question)
    session.run()


📚 Начинаем обучение

📌 Тема: Какие причины провала стартапа?


──────────────────────────────────────────────────
📊 Стратегия: socratic

🧠 Тьютор: Какие факторы, по вашему мнению, могут повлиять на успешность стартапа в условиях конкуренции? Как вы думаете, что может произойти, если стартап не учитывает потребности рынка?

💬 Намерение: answer

📊 Оценка: score=0.00 | Студенту следует ознакомиться с материалами курса, чтобы понять основные причины провала стартапов.

──────────────────────────────────────────────────
📊 Стратегия: socratic

🧠 Тьютор: Как вы думаете, какие последствия могут возникнуть для стартапа, если он не сможет привлечь достаточное внимание и интерес со стороны целевой аудитории? Какие факторы могут способствовать тому, чтобы продукт стал востребованным на рынке?

💬 Намерение: answer

📊 Оценка: score=0.00 | Ответ студента не соответствует требованиям задания, так как он не содержит информации о причинах провала стартапов.

────────────────────────────────────────────

In [ ]:
import os
import json
from dataclasses import dataclass, field
from typing import List, Optional

from openai import OpenAI
from qdrant_client import QdrantClient

client = OpenAI(api_key="")  # берет ключ из ENV

EMBED_MODEL = "text-embedding-3-large"
CHAT_MODEL = "gpt-4o-mini"

QDRANT_URL = "https://qdrant.dev.adapstory.com"
COLLECTION_NAME = "presentations_industrix_openai"

os.environ["QDRANT_DISABLE_CHECK"] = "1"

qdrant = QdrantClient(url=QDRANT_URL, port=443, timeout=120)


# ─────────────────────────── helpers ────────────────────────────

def embed_query(text: str) -> list:
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding


def ask_llm(prompt: str, temperature: float = 0.3) -> str:
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=700,
    )
    return response.choices[0].message.content.strip()


def parse_json(text: str) -> dict:
    """Безопасный парсинг JSON — убирает markdown-обёртки."""
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(cleaned)


def retrieve_context(question: str, top_k: int = 10) -> list:
    vector = embed_query(question)
    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=vector,
        limit=top_k,
        with_payload=True,
    )
    contexts = []
    for r in results:
        payload = r.payload or {}
        text = payload.get("text", "").strip()
        if not text:
            continue
        contexts.append({
            "text": text,
            "page": payload.get("page"),
            "source": payload.get("source"),
            "score": r.score,
        })
    return contexts


def build_context(contexts: list) -> str:
    chunks = []
    for i, c in enumerate(contexts):
        chunks.append(
            f"CHUNK_ID: {i}\nSOURCE: {c.get('source')}\nPAGE: {c.get('page')}\n\nTEXT:\n{c.get('text')}"
        )
    return "\n\n---\n\n".join(chunks)


def format_history(history: list) -> str:
    return "\n".join(history) if history else "История диалога пуста."


# ─────────────────────────── state ──────────────────────────────

@dataclass
class StudentState:
    topic: str
    attempts: int = 0
    mastery_score: float = 0.0
    history: List[str] = field(default_factory=list)
    misconceptions: List[str] = field(default_factory=list)
    last_strategy: Optional[str] = None
    explanation_used: bool = False
    consecutive_wrong: int = 0
    consecutive_chat: int = 0   # сколько раз подряд студент уклонился от ответа


# ─────────────────────────── question type ──────────────────────
# НОВОЕ: классификатор типа вопроса — learning vs organizational

class QuestionTypeClassifier:
    """
    Определяет тип вопроса ДО запуска учебной сессии.

    organizational — вопрос про процесс курса (домашки, дедлайны,
                     обратная связь, расписание, оценки и т.п.)
    learning       — вопрос про содержание курса (термины, концепции)
    other          — всё остальное
    """

    def classify(self, question: str) -> str:
        prompt = f"""
Ты классифицируешь вопросы студентов в учебном курсе.

Вопрос: "{question}"

Категории и их ПРИЗНАКИ:

organizational — вопрос про ПРОЦЕСС обучения, а НЕ про содержание.
  Ключевые слова: домашка, задание, дедлайн, обратная связь, преподаватель,
  расписание, оценки, платформа, проверка, сдача, срок, вебинар, чат, ментор, эксперт, формат сдачи, домашенее задание, как сдвать.
  Примеры: "Будет ли обратная связь?", "Когда дедлайн?",
            "Как сдавать домашку?", "Есть ли проверка заданий?"

learning — вопрос про содержание курса: термины, концепции, темы.
  Примеры: "Что такое юнит-экономика?", "Почему стартапы проваливаются?"

other — всё остальное, не связанное с курсом.

ЖЁСТКОЕ ПРАВИЛО: если в вопросе есть хотя бы одно из слов:
домашка / задание / дедлайн / обратная связь / преподаватель /
расписание / проверка / вебинар / чат — это ВСЕГДА organizational.

Верни ТОЛЬКО JSON без пояснений: {{"type": "learning|organizational|other"}}
"""
        try:
            res = ask_llm(prompt, temperature=0)
            return parse_json(res).get("type", "learning")
        except Exception:
            return "learning"


class OrganizationalAgent:
    """
    Отвечает на организационные вопросы ТОЛЬКО из контекста RAG.
    Не придумывает ответы из общих знаний.
    """

    def respond(self, question: str, context: str) -> str:
        prompt = f"""
Студент курса задал организационный вопрос.

Информация из материалов курса:
{context}

Вопрос студента: "{question}"

Правила:
- Отвечай ТОЛЬКО на основе информации из материалов курса выше
- Если информация есть — изложи её кратко и точно своими словами (2-4 предложения)
- Если информации нет — скажи: "В материалах курса нет информации по этому вопросу. Уточните у преподавателя."
- Не придумывай ответы из общих знаний LLM
"""
        return ask_llm(prompt, temperature=0.1)


# ─────────────────────────── agents ─────────────────────────────

class PlannerAgent:
    """
    Стратегии:
      socratic  — задаём наводящий вопрос
      hint      — даём подсказку
      explain   — объясняем тему
      verify    — после объяснения проверяем понимание
    """

    def plan(
        self,
        question: str,
        context_text: str,
        state: StudentState,
        last_answer: Optional[str] = None,
        evaluation: Optional[dict] = None,
    ) -> dict:

        misconceptions_text = (
            "\n".join(f"- {m}" for m in state.misconceptions)
            if state.misconceptions
            else "нет"
        )

        prompt = f"""
Ты педагогический планировщик ИИ-тьютора.

Выбери ОДНУ стратегию из: socratic, hint, explain, verify.

Правила выбора:
1. socratic  — студент ещё не пробовал отвечать или ответил частично (score < 0.5), attempts <= 1
2. hint      — студент ответил, но score < 0.6, attempts <= 3; или есть явное заблуждение
3. explain   — attempts >= 3 ИЛИ consecutive_wrong >= 2 ИЛИ студент явно не понимает
4. verify    — стратегия была explain → нужно проверить понимание (задать короткий вопрос)
5. НЕ повторяй одну стратегию больше 2 раз подряд без изменений
6. Если score > 0.8 → ответь "mastered" (тема усвоена, сессию можно завершать)

Текущее состояние:
- attempts: {state.attempts}
- consecutive_wrong: {state.consecutive_wrong}
- last_strategy: {state.last_strategy or "нет"}
- explanation_used: {state.explanation_used}
- mastery_score: {state.mastery_score}

Заблуждения студента:
{misconceptions_text}

Последний ответ студента:
{last_answer or "пока нет ответа"}

Оценка последнего ответа:
{json.dumps(evaluation, ensure_ascii=False) if evaluation else "нет"}

Контекст (фрагмент):
{context_text[:800]}

Верни ТОЛЬКО JSON без пояснений:
{{"strategy": "socratic|hint|explain|verify|mastered"}}
"""

        response = ask_llm(prompt, temperature=0.1)

        try:
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            plan = json.loads(cleaned)
            strategy = plan.get("strategy", "socratic")
        except Exception:
            strategy = "socratic"

        return {"strategy": strategy}


class TutorAgent:

    def respond(
        self,
        strategy: str,
        question: str,
        contexts: list,
        history: list,
        misconceptions: Optional[List[str]] = None,
    ) -> str:

        context_text = build_context(contexts)
        history_text = format_history(history)
        misc_text = (
            "Заблуждения студента, которые нужно исправить:\n"
            + "\n".join(f"- {m}" for m in misconceptions)
            if misconceptions
            else ""
        )

        strategy_instructions = {
            "socratic": (
                "Задай ОДИН наводящий вопрос, который помогает студенту самому прийти к ответу.\n"
                "Не давай ответ и не перечисляй факты. Максимум — 2 предложения."
            ),
            "hint": (
                "Дай ОДНУ небольшую подсказку, указывающую направление мысли.\n"
                "Не раскрывай полный ответ. Упомяни заблуждение студента, если оно есть.\n"
                "Максимум — 3 предложения."
            ),
            "explain": (
                "Объясни тему развёрнуто и структурированно.\n"
                "Используй ТОЛЬКО контекст курса.\n"
                "В конце укажи источник: [Источник: SOURCE, стр. PAGE]\n"
                "Исправь заблуждения студента, если они есть."
            ),
            "verify": (
                "Ты только что объяснил тему. Теперь задай студенту ОДИН короткий проверочный вопрос,\n"
                "чтобы убедиться, что он понял объяснение. Вопрос должен быть конкретным."
            ),
        }

        instruction = strategy_instructions.get(strategy, strategy_instructions["socratic"])

        prompt = f"""
Ты ИИ-тьютор курса. Используй ТОЛЬКО информацию из контекста.
Если информации нет — скажи: «В материалах курса нет информации по этому вопросу».

КОНТЕКСТ КУРСА:
{context_text}

ВОПРОС СТУДЕНТА:
{question}

ИСТОРИЯ ДИАЛОГА:
{history_text}

{misc_text}

СТРАТЕГИЯ: {strategy}
ИНСТРУКЦИЯ:
{instruction}

ОТВЕТ ТЬЮТОРА:
"""

        return ask_llm(prompt, temperature=0.3)


class IntentClassifier:
    """
    Определяет намерение сообщения студента:
      answer    — попытка ответить на вопрос тьютора
      chat      — разговорное сообщение (спасибо, понял, окей и т.д.)
      question  — студент задаёт свой вопрос по теме
      off_topic — сообщение не по теме
    """

    def classify(self, student_message: str, tutor_last_message: str, topic: str) -> str:

        prompt = f"""
Ты классификатор намерений в диалоге тьютора и студента.

Тема обучения: {topic}

Последнее сообщение тьютора:
{tutor_last_message}

Сообщение студента:
{student_message}

Определи намерение сообщения студента. Варианты:
- answer     — студент пытается ответить на вопрос тьютора (даже частично или неверно)
- chat       — разговорное сообщение: благодарность, подтверждение, "понял", "окей", "спасибо" и т.п.
- question   — студент задаёт свой вопрос по теме курса
- off_topic  — сообщение не связано с темой обучения

Верни ТОЛЬКО JSON:
{{"intent": "answer|chat|question|off_topic"}}
"""

        response = ask_llm(prompt, temperature=0)

        try:
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            return json.loads(cleaned).get("intent", "answer")
        except Exception:
            return "answer"


class ChatAgent:
    """
    Отвечает на разговорные сообщения и вопросы студента,
    после чего мягко возвращает к теме обучения.
    """

    def respond_to_chat(
        self,
        student_message: str,
        tutor_last_message: str,
        topic: str,
        history: list,
    ) -> str:

        history_text = format_history(history[-6:])

        prompt = f"""
Ты ИИ-тьютор курса. Студент написал тебе разговорное сообщение.

Тема обучения: {topic}

История диалога:
{history_text}

Твоё последнее сообщение:
{tutor_last_message}

Сообщение студента:
{student_message}

Ответь естественно и коротко (1-2 предложения).
Затем мягко напомни свой предыдущий вопрос или предложи продолжить.
Не повторяй весь вопрос дословно — перефразируй кратко.
"""

        return ask_llm(prompt, temperature=0.4)

    def respond_to_question(
        self,
        student_question: str,
        contexts: list,
        topic: str,
        history: list,
        tutor_last_message: str,
    ) -> str:

        context_text = build_context(contexts[:3])
        history_text = format_history(history[-6:])

        prompt = f"""
Ты ИИ-тьютор курса. Студент задал уточняющий вопрос.

Тема обучения: {topic}
Контекст курса: {context_text}

История диалога:
{history_text}

Вопрос студента: {student_question}

Ответь кратко, используя только контекст курса (2-4 предложения).
Если информации нет — скажи честно.
После ответа мягко верни студента к теме, напомнив свой вопрос одним предложением.

Твой предыдущий вопрос был: {tutor_last_message}
"""

        return ask_llm(prompt, temperature=0.3)


class EvaluatorAgent:

    def evaluate(self, question: str, student_answer: str, context: str) -> dict:

        prompt = f"""
Ты оцениваешь ответ студента на вопрос по материалам курса.

Вопрос: {question}
Ответ студента: {student_answer}
Контекст курса: {context}

Верни ТОЛЬКО JSON без пояснений:
{{
  "correct": true/false,
  "partial": true/false,
  "misconception": "описание заблуждения или пустая строка",
  "score": 0.0-1.0,
  "feedback": "краткий комментарий для тьютора (не для студента)"
}}
"""

        response = ask_llm(prompt, temperature=0)

        try:
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            return json.loads(cleaned)
        except Exception:
            return {"correct": False, "partial": False, "misconception": "", "score": 0.0, "feedback": ""}


# ─────────────────────────── orchestrator ───────────────────────

class TutorOrchestrator:

    MAX_ATTEMPTS = 6

    def __init__(self, question: str):
        self.question = question
        self.contexts = retrieve_context(question)
        self.context_text = build_context(self.contexts[:5])

        self.state = StudentState(topic=question)

        # НОВОЕ: агенты для организационных вопросов
        self.qtype_clf = QuestionTypeClassifier()
        self.org_agent = OrganizationalAgent()

        self.planner = PlannerAgent()
        self.tutor = TutorAgent()
        self.evaluator = EvaluatorAgent()
        self.intent_clf = IntentClassifier()
        self.chat_agent = ChatAgent()

        self._last_tutor_msg = ""

    def _update_state_after_eval(self, evaluation: dict, last_answer: str):
        score = evaluation.get("score", 0.0)
        self.state.mastery_score = score
        self.state.attempts += 1

        misconception = evaluation.get("misconception", "")
        if misconception and misconception not in self.state.misconceptions:
            self.state.misconceptions.append(misconception)

        if not evaluation.get("correct") and score < 0.5:
            self.state.consecutive_wrong += 1
        else:
            self.state.consecutive_wrong = 0

        self.state.history.append(f"Студент: {last_answer}")

    def _print_sep(self):
        print("\n" + "─" * 50)

    def run(self):

        # ══════════════════════════════════════════
        # ШАГ 0: ОПРЕДЕЛЯЕМ ТИП ВОПРОСА
        # ══════════════════════════════════════════

        qtype = self.qtype_clf.classify(self.question)
        print(f"🔎 Тип вопроса: {qtype}")

        # Организационный вопрос → отвечаем из RAG и завершаем
        if qtype == "organizational":
            if not self.contexts:
                print("\n🧠 Тьютор: В материалах курса нет информации по этому вопросу. Уточните у преподавателя.")
            else:
                print("\n🧠 Тьютор:", self.org_agent.respond(self.question, self.context_text))
            print("\n✅ Завершено")
            return

        # Учебный вопрос — проверяем базу знаний
        if not self.contexts:
            print("❌ Нет информации в базе знаний по данному вопросу.")
            return

        print("\n📚 Начинаем обучение\n")
        print(f"📌 Тема: {self.question}\n")

        last_answer = None
        evaluation = None
        force_next_strategy = None

        while True:

            # ════════════════════════════════════════════
            # ФАЗА 1: ТЬЮТОР ГОВОРИТ
            # ════════════════════════════════════════════

            if force_next_strategy:
                strategy = force_next_strategy
                force_next_strategy = None
            else:
                plan = self.planner.plan(
                    self.question,
                    self.context_text,
                    self.state,
                    last_answer,
                    evaluation,
                )
                strategy = plan["strategy"]

            if strategy == "mastered":
                print("\n✅ Отлично! Тьютор подтверждает: тема усвоена.")
                break

            if self.state.attempts >= self.MAX_ATTEMPTS:
                print("\n⚠️  Достигнут лимит попыток. Завершаем сессию.")
                break

            self._print_sep()
            print(f"📊 Стратегия: {strategy}")

            tutor_msg = self.tutor.respond(
                strategy,
                self.question,
                self.contexts,
                self.state.history,
                self.state.misconceptions,
            )
            print(f"\n🧠 Тьютор: {tutor_msg}")

            self.state.last_strategy = strategy
            self.state.history.append(f"Тьютор: {tutor_msg}")
            self._last_tutor_msg = tutor_msg

            # После explain → следующий ход ВСЕГДА verify (детерминировано)
            if strategy == "explain":
                self.state.explanation_used = True
                force_next_strategy = "verify"

            # ════════════════════════════════════════════
            # ФАЗА 2: СТУДЕНТ ГОВОРИТ
            # (внутренний цикл — chat/question/off_topic крутятся здесь)
            # ════════════════════════════════════════════

            while True:
                print()
                raw_input = input("👨‍🎓 Студент: ").strip()

                if not raw_input:
                    print("(тьютор ждёт ответа...)")
                    continue

                intent = self.intent_clf.classify(
                    raw_input,
                    self._last_tutor_msg,
                    self.question,
                )
                print(f"💬 Намерение: {intent}")

                # chat ─────────────────────────────────────────
                if intent == "chat":
                    reply = self.chat_agent.respond_to_chat(
                        raw_input,
                        self._last_tutor_msg,
                        self.question,
                        self.state.history,
                    )
                    print(f"\n🧠 Тьютор: {reply}")

                    # После explain/verify "спасибо" = сессия завершена
                    if self.state.explanation_used:
                        print("\n✅ Сессия завершена. Удачи в обучении!")
                        return

                    self.state.history.append(f"Студент: {raw_input}")
                    self.state.history.append(f"Тьютор: {reply}")
                    self._last_tutor_msg = reply
                    self.state.consecutive_chat += 1

                    # Студент 2 раза уклонился — принудительно двигаемся дальше
                    if self.state.consecutive_chat >= 2:
                        self.state.consecutive_chat = 0
                        self.state.consecutive_wrong += 1  # засчитываем как неудачную попытку
                        print("\n🧠 Тьютор: Похоже, тема даётся сложно — давай я объясню сам.")
                        # Выходим из фазы 2, планировщик выберет explain
                        break

                    continue

                # question ─────────────────────────────────────
                if intent == "question":
                    reply = self.chat_agent.respond_to_question(
                        raw_input,
                        self.contexts,
                        self.question,
                        self.state.history,
                        self._last_tutor_msg,
                    )
                    print(f"\n🧠 Тьютор: {reply}")
                    self.state.history.append(f"Студент: {raw_input}")
                    self.state.history.append(f"Тьютор: {reply}")
                    self._last_tutor_msg = reply
                    continue

                # off_topic ────────────────────────────────────
                if intent == "off_topic":
                    print(f"\n🧠 Тьютор: Давай вернёмся к теме. {self._last_tutor_msg}")
                    continue

                # answer → выходим из фазы 2 ──────────────────
                break

            # ════════════════════════════════════════════
            # ФАЗА 3: ОЦЕНИВАЕМ ОТВЕТ
            # ════════════════════════════════════════════

            last_answer = raw_input

            evaluation = self.evaluator.evaluate(
                self.question,
                last_answer,
                self.context_text,
            )

            score = evaluation.get("score", 0.0)
            feedback = evaluation.get("feedback", "")
            print(f"\n📊 Оценка: score={score:.2f} | {feedback}")

            self._update_state_after_eval(evaluation, last_answer)

            if evaluation.get("correct") or score >= 0.85:
                print("\n✅ Правильно! Тема усвоена.")
                break


# ─────────────────────────── entry point ────────────────────────

if __name__ == "__main__":
    question = "Какие вообще модули и этапы курса"
    TutorOrchestrator(question).run()

🔎 Тип вопроса: organizational

🧠 Тьютор: Курс состоит из двух этапов. Первый этап включает 7 образовательных модулей, таких как знакомство с ГПН, коммерциализация и Lean-подход, разведка заказчика и другие. Второй этап включает 8 модулей, среди которых поиск и разведка бизнес-заказчика, управление рисками, самодиагностика проекта, подготовка к пилоту и масштабирование проекта.

✅ Завершено
